In [0]:
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window
from utils.logger import get_logger

In [0]:
logger = get_logger("gold_layer")

## **Load Silver Data**

In [0]:
try:

    logger.info("Loading Silver table")

    silver_df = spark.table("dbdemos.silver.healthcare_qa_clean")

    logger.info("Silver data loaded successfully")

except Exception as e:

    logger.error(f"Failed to load Silver table: {str(e)}")
    raise

## **Populate Product Dimension**

In [0]:
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

try:

    logger.info("Creating dim_product")

    product_dim = (
        silver_df
        .select("product", "product_grade")
        .distinct()
    )

    window_spec = Window.orderBy("product")

    product_dim = product_dim.withColumn(
        "product_key",
        row_number().over(window_spec).cast("bigint")
    )

    product_dim = product_dim.select(
        "product_key",
        "product",
        "product_grade"
    )

    product_dim.write.mode("overwrite").saveAsTable(
        "dbdemos.gold.dim_product"
    )

    logger.info("dim_product populated successfully")

except Exception as e:

    logger.error(f"dim_product creation failed: {str(e)}")
    raise

## **Populate Method Dimension**

In [0]:
try:

    logger.info("Creating dim_method")

    method_dim = (
        silver_df
        .select("x_method", "stage", "units")
        .distinct()
    )

    window_spec = Window.orderBy("x_method")

    method_dim = method_dim.withColumn(
        "method_key",
        row_number().over(window_spec).cast("bigint")
    )

    method_dim = method_dim.select(
        "method_key",
        "x_method",
        "stage",
        "units"
    )

    method_dim.write.mode("overwrite").saveAsTable(
        "dbdemos.gold.dim_method"
    )

    logger.info("dim_method populated successfully")

except Exception as e:

    logger.error(f"dim_method creation failed: {str(e)}")
    raise

## **Populate Status Dimension**

In [0]:
try:

    logger.info("Creating dim_status")

    status_dim = (
        silver_df
        .select("status", "reportable", "x_disposed")
        .distinct()
    )

    window_spec = Window.orderBy("status")

    status_dim = status_dim.withColumn(
        "status_key",
        row_number().over(window_spec).cast("bigint")
    )

    status_dim = status_dim.select(
        "status_key",
        "status",
        "reportable",
        "x_disposed"
    )

    status_dim.write.mode("overwrite").saveAsTable(
        "dbdemos.gold.dim_status"
    )

    logger.info("dim_status populated successfully")

except Exception as e:

    logger.error(f"dim_status creation failed: {str(e)}")
    raise

## **Populate Batch Dimension**

In [0]:
try:

    logger.info("Creating dim_batch")

    batch_dim = (
        silver_df
        .select("lot_name", "lot_number")
        .distinct()
    )

    window_spec = Window.orderBy("lot_name")

    batch_dim = batch_dim.withColumn(
        "batch_key",
        row_number().over(window_spec).cast("bigint")
    )

    batch_dim = batch_dim.select(
        "batch_key",
        "lot_name",
        "lot_number"
    )

    batch_dim.write.mode("overwrite").saveAsTable(
        "dbdemos.gold.dim_batch"
    )

    logger.info("dim_batch populated successfully")

except Exception as e:

    logger.error(f"dim_batch creation failed: {str(e)}")
    raise

## **Populate Batch Dimension**

In [0]:
try:

    logger.info("Creating dim_test")

    test_dim = (
        silver_df
        .select(
            "test_number",
            "name",
            "item_description",
            "reported_name"
        )
        .distinct()
    )

    window_spec = Window.orderBy("test_number")

    test_dim = test_dim.withColumn(
        "test_key",
        row_number().over(window_spec).cast("bigint")
    )

    test_dim = test_dim.select(
        "test_key",
        "test_number",
        "name",
        "item_description",
        "reported_name"
    )

    test_dim.write.mode("overwrite").saveAsTable(
        "dbdemos.gold.dim_test"
    )

    logger.info("dim_test populated successfully")

except Exception as e:

    logger.error(f"dim_test creation failed: {str(e)}")
    raise

## **Load Dimension Tables**

In [0]:
def load_table(table_name):

    try:

        logger.info(f"Loading table {table_name}")

        df = spark.table(table_name)

        logger.info(f"{table_name} loaded successfully")

        return df

    except Exception as e:

        logger.error(f"Failed to load {table_name}: {str(e)}")
        raise

In [0]:
dim_product = load_table("dbdemos.gold.dim_product")
dim_method = load_table("dbdemos.gold.dim_method")
dim_status = load_table("dbdemos.gold.dim_status")
dim_batch = load_table("dbdemos.gold.dim_batch")
dim_test = load_table("dbdemos.gold.dim_test")

## **Write Fact Table**

In [0]:
try:

    logger.info("Creating fact_qa_tests")

    fact_df = (
        silver_df
        .join(dim_product, ["product", "product_grade"], "left")
        .join(dim_method, ["x_method", "stage", "units"], "left")
        .join(dim_status, ["status", "reportable", "x_disposed"], "left")
        .join(dim_batch, ["lot_name", "lot_number"], "left")
        .join(dim_test, ["test_number", "name", "item_description", "reported_name"], "left")
    )

    fact_df = fact_df.select(
        silver_df.sample_number.alias("test_id"),
        "product_key",
        "method_key",
        "status_key",
        "batch_key",
        "test_key",
        "label_id",
        "sample_number",
        "entry",
        "result_number",
        "description",
        "date_completed",
        "sampled_date"
    )

    fact_df.write.mode("overwrite").saveAsTable(
        "dbdemos.gold.fact_qa_tests"
    )

    logger.info("fact_qa_tests populated successfully")

except Exception as e:

    logger.error(f"fact_qa_tests creation failed: {str(e)}")
    raise

In [0]:
%sql
select count(*) from dbdemos.gold.fact_qa_tests